# Akan-BPE — 2A5 tiny-aya-base BPB diagnostic & truncation-corrected eval

Sibling of `2a4_llama_eval_helper.ipynb`, for **Phase 2A5** (`CohereLabs/tiny-aya-base`, 3.35B,
Africa-aware). Like Llama, 2A5 shows a negative raw BPB (mean-subword **0.8949** vs base
**0.8779**, −0.0170, report §12) — and like Llama it is one of the **two highest-base-fertility**
rungs (2.975 tok/word), the fingerprint of a **truncation bias**:

- `_build_causal_example` truncates each text to `max_length-1` tokens, but `compute_model_bpb`
  divides the (truncated) NLL by the **full** UTF-8 byte count.
- The eval texts are paragraphs, so longer-than-budget texts have their BPB **underestimated —
  worse for higher-fertility tokenizers**. tiny-aya's base tokenizer (~2.98 tok/word) covers only
  ~85 words of a 256-token window vs ~202 for the 1.26-fertility Akan tokenizer, deflating its base
  BPB.

**Plan:** (A) setup → (B) load eval + tokenizers → (C) quantify truncation →
(D) reproduce the 0.8779 artifact → (E) corrected **base** BPB (full byte coverage, no adapters) →
(F) retrain both arms in-session → (G) corrected **experiment** BPB + corrected improvement →
(H) summary / verdict.

Runs top-to-bottom on a Kaggle single T4. `CohereLabs/tiny-aya-base` is CC-BY-NC gated — accept the
terms and provide an HF token (cell 3). 3.35B in 4-bit is tighter than the ~1B rungs but fits.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/attabeezy/akan-bpe/blob/main/notebooks/2a5_tiny_aya_eval_helper.ipynb)  [![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/2a5_tiny_aya_eval_helper.ipynb)

## A. Setup

In [ ]:
# 1. Clone repo (skip if already inside it)
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

In [ ]:
# 2. Install dependencies
%pip install -q -e ".[dev,train]"
%pip install -q bitsandbytes sentencepiece pandas

In [ ]:
# Hugging Face authentication - tiny-aya-base requires accepting CC-BY-NC terms.
# Accept at https://huggingface.co/CohereLabs/tiny-aya-base, then run this cell and
# paste a token with access (or set the HF_TOKEN env var / Kaggle Secret).
from huggingface_hub import login
login()

In [ ]:
# 3. Confirm GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime / Settings -> Accelerator -> GPU T4, then re-run."
    )
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Download Akan datasets from HuggingFace Hub
# --tts-limit 50000 pins the pristine-Twi corpus to the same 45,000/2,500/2,500
# split used in the report (the default would pull ~188k rows and shift the split).
!python scripts/download.py --output-dir data --tts-limit 50000

In [ ]:
# 5. Train TTS tokenizer (if not already present)
from pathlib import Path

if not Path("models/tts_tokenizer.json").exists():
    print("TTS tokenizer not found - training now ...")
    !python scripts/train_bpe.py --inputs data/pristine_twi_train.jsonl --output models/tts_tokenizer.json --name tts
else:
    sz = Path("models/tts_tokenizer.json").stat().st_size / 1e6
    print(f"TTS tokenizer already present ({sz:.1f} MB), skipping.")

In [ ]:
# 6. Verify all required inputs exist
from pathlib import Path

required = {
    "TTS train data": Path("data/pristine_twi_train.jsonl"),
    "TTS test data": Path("data/pristine_twi_test.jsonl"),
    "TTS tokenizer": Path("models/tts_tokenizer.json"),
}
missing = [name for name, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")
print("All inputs ready.")
for name, p in required.items():
    print(f"  {name}: {p}  ({p.stat().st_size / 1e6:.1f} MB)")

## B. Load eval texts + tokenizers

The full **2,500-row** pristine-Twi test set (the 2A5 run only scored a 512-row cap).

In [ ]:
import math
import numpy as np
import pandas as pd

from akan_bpe.model_integration import (
    load_texts,
    load_experiment_tokenizer,
    compute_model_bpb,        # original (truncating) BPB - used to reproduce the artifact
    build_text_dataset,       # original fixed-width dataset builder (truncates)
    _load_base_model_for_bpb, # quantized base-model loader
    _import_training_stack,   # torch + AutoModelForCausalLM + BitsAndBytesConfig + PeftModel
    _set_model_token_config,
)
from akan_bpe.metrics import count_utf8_bytes, bits_per_byte
from transformers import AutoTokenizer

stack = _import_training_stack()
torch = stack["torch"]
AutoModelForCausalLM = stack["AutoModelForCausalLM"]
BitsAndBytesConfig = stack["BitsAndBytesConfig"]
PeftModel = stack["PeftModel"]

MAX_LENGTH = 256   # the token budget used by the original runs (truncating path)
CHUNK_SIZE = 256   # full-coverage scoring chunk size (matches training context)

AYA_ID = "CohereLabs/tiny-aya-base"
EVAL_FILE = Path("data/pristine_twi_test.jsonl")
AKAN_TOK_PATH = Path("models/tts_tokenizer.json")

eval_texts = load_texts(EVAL_FILE)                 # full 2,500 rows, untruncated
EVAL_BYTES = count_utf8_bytes(eval_texts)
print(f"Eval texts        : {len(eval_texts):,}")
print(f"Total UTF-8 bytes : {EVAL_BYTES:,}")

akan_tok = load_experiment_tokenizer(AKAN_TOK_PATH)
aya_base_tok = AutoTokenizer.from_pretrained(AYA_ID)
# pad_token fallback mirrors compute_bpb_metrics in the library (line 325-326).
if aya_base_tok.pad_token is None:
    aya_base_tok.pad_token = aya_base_tok.eos_token or aya_base_tok.unk_token
print("Loaded Akan + tiny-aya-base tokenizers.")

## C. Truncation diagnosis (no model load)

Per tokenizer over the **full** eval: token-length distribution, the **fraction of texts
truncated** at `max_length=256`, and the **covered-byte fraction** — the share of each text's bytes
the original (truncating) BPB actually scored. A low covered-byte fraction means the original BPB
divided real NLL by bytes it never scored, deflating it.

In [ ]:
def truncation_stats(tokenizer, texts, max_length):
    """Token-length distribution + truncation/coverage stats for one tokenizer."""
    budget = max_length - 1  # _build_causal_example keeps max_length-1 tokens, then EOS
    tok_lens = []
    covered_bytes = 0
    full_bytes = 0
    for t in texts:
        ids = tokenizer(t, add_special_tokens=False)["input_ids"]
        tok_lens.append(len(ids))
        fb = len(t.encode("utf-8"))
        full_bytes += fb
        if len(ids) > budget:
            cov_text = tokenizer.decode(ids[:budget])
            covered_bytes += min(len(cov_text.encode("utf-8")), fb)
        else:
            covered_bytes += fb
    a = np.array(tok_lens)
    return {
        "mean_tokens": round(float(a.mean()), 1),
        "median_tokens": int(np.median(a)),
        "p90_tokens": int(np.percentile(a, 90)),
        "max_tokens": int(a.max()),
        "pct_truncated": round(100.0 * float((a > budget).mean()), 1),
        "covered_byte_frac": round(covered_bytes / full_bytes, 4) if full_bytes else 0.0,
    }


diag = pd.DataFrame({
    "Akan-TTS": truncation_stats(akan_tok, eval_texts, MAX_LENGTH),
    "tiny-aya-base": truncation_stats(aya_base_tok, eval_texts, MAX_LENGTH),
}).T
print("Truncation diagnosis at max_length =", MAX_LENGTH)
diag

### Full-coverage BPB helper

`compute_model_bpb_full` scores **100% of every text's bytes** via consecutive non-overlapping
token chunks (no truncation). It reuses the same causal-shift sum-NLL math as
`akan_bpe.model_integration.compute_model_bpb` and the same `count_utf8_bytes` / `bits_per_byte`
primitives, so it is consistent with the library metric — only the coverage differs. Identical to
the helper in `2a4_llama_eval_helper.ipynb`.

In [ ]:
def compute_model_bpb_full(model, texts, tokenizer, torch, chunk_size=256):
    """Full byte-coverage BPB via non-overlapping chunks.

    Every text's bytes appear in the denominator. ~1/chunk_size boundary tokens
    per chunk have no left-context (cold start); this effect is symmetric across
    models. Single-token final chunks (len(ids) % chunk_size == 1) are skipped
    — affects ~0.4% of texts symmetrically across models.
    Use compute_model_bpb_sliding for a strided alternative without cold-start bias.
    """
    cross_entropy = torch.nn.CrossEntropyLoss(reduction="sum")
    total_nll_nats = 0.0
    num_target_tokens = 0
    eos_id = tokenizer.eos_token_id
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            if eos_id is not None:
                ids = ids + [eos_id]
            for start in range(0, len(ids), chunk_size):
                chunk = ids[start:start + chunk_size]
                if len(chunk) < 2:
                    continue  # single orphan token: ~0.4% of texts, symmetric
                input_ids = torch.tensor([chunk], device=model.device)
                outputs = model(input_ids=input_ids)
                shift_logits = outputs.logits[:, :-1, :]
                shift_labels = input_ids[:, 1:]
                loss = cross_entropy(
                    shift_logits.reshape(-1, shift_logits.size(-1)).float(),
                    shift_labels.reshape(-1),
                )
                total_nll_nats += float(loss.item())
                num_target_tokens += int(shift_labels.numel())
    total_bytes = count_utf8_bytes(texts)
    return {
        "bits_per_byte": bits_per_byte(total_nll_nats, total_bytes),
        "total_nll_bits": total_nll_nats / math.log(2),
        "total_bytes": total_bytes,
        "num_target_tokens": num_target_tokens,
    }


def compute_model_bpb_sliding(model, texts, tokenizer, torch, window=512, stride=256):
    """Strided sliding-window BPB: each target scored exactly once.

    Every new target has at least (window - stride - 1) tokens of left context
    within its window, eliminating the chunk-boundary cold-start of
    compute_model_bpb_full. window=512, stride=256 gives >=255 tokens of context
    per new target. 100% of bytes appear in the denominator; each target position
    is counted exactly once (no double-counting).
    """
    cross_entropy = torch.nn.CrossEntropyLoss(reduction="sum", ignore_index=-100)
    total_nll_nats = 0.0
    num_target_tokens = 0
    eos_id = tokenizer.eos_token_id
    model.eval()
    with torch.no_grad():
        for text in texts:
            ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            if eos_id is not None:
                ids = ids + [eos_id]
            n = len(ids)
            if n < 2:
                continue
            # scored_end: exclusive upper bound of target positions already counted.
            # Target positions in ids are [1, n); scored_end starts at 1 (nothing counted).
            scored_end = 1
            for begin in range(0, n - 1, stride):
                end = min(begin + window, n)
                chunk = ids[begin:end]
                if len(chunk) < 2:
                    break
                inp = torch.tensor([chunk], device=model.device)
                outputs = model(input_ids=inp)
                shift_logits = outputs.logits[:, :-1, :]
                # shift_labels[0, j] is the target for ids[begin + j + 1].
                shift_labels = inp[:, 1:].clone()
                # Mask targets already counted in a previous window.
                n_mask = max(0, scored_end - begin - 1)
                if n_mask > 0:
                    shift_labels[:, :n_mask] = -100
                valid = int((shift_labels != -100).sum().item())
                if valid > 0:
                    loss = cross_entropy(
                        shift_logits.reshape(-1, shift_logits.size(-1)).float(),
                        shift_labels.reshape(-1),
                    )
                    total_nll_nats += float(loss.item())
                    num_target_tokens += valid
                scored_end = end
                if end == n:
                    break
    total_bytes = count_utf8_bytes(texts)
    return {
        "bits_per_byte": bits_per_byte(total_nll_nats, total_bytes),
        "total_nll_bits": total_nll_nats / math.log(2),
        "total_bytes": total_bytes,
        "num_target_tokens": num_target_tokens,
    }


def free():
    """Collect garbage and clear the CUDA cache. Caller must `del` its own model
    reference *before* calling this."""
    import gc
    gc.collect()
    torch.cuda.empty_cache()

## D. Reproduce the 0.8779 artifact

Score base tiny-aya with the **original truncating** `compute_model_bpb` on the same capped config
as the run (first **512** texts, `max_length=256`). This should reproduce the reported base BPB of
~**0.8779**, confirming the notebook faithfully replicates the pipeline before we correct it.

In [ ]:
aya_base = _load_base_model_for_bpb(AYA_ID, AutoModelForCausalLM, torch, quantized=True)
_set_model_token_config(aya_base, aya_base_tok)  # align pad/eos/bos config (mirrors compute_bpb_metrics)

repro_texts = eval_texts[:512]
repro_ds = build_text_dataset(repro_texts, aya_base_tok, MAX_LENGTH)
repro = compute_model_bpb(aya_base, repro_ds, count_utf8_bytes(repro_texts), torch)
print(f"Reproduced base tiny-aya BPB (512 rows, trunc max_length=256): {repro.bits_per_byte:.4f}")
print("Report  base tiny-aya BPB (2A5, same capped config)           : 0.8779")

### Sanity check: full-coverage equals the library metric when nothing truncates

On texts that fit within the token budget for both tokenizers, no truncation happens, so
`compute_model_bpb_full` must equal `compute_model_bpb`.

In [ ]:
budget = MAX_LENGTH - 1
short_texts = [
    t for t in eval_texts[:400]
    if len(akan_tok(t, add_special_tokens=False)["input_ids"]) <= budget
    and len(aya_base_tok(t, add_special_tokens=False)["input_ids"]) <= budget
][:64]
print(f"Short non-truncated texts used for the check: {len(short_texts)}")

old_ds = build_text_dataset(short_texts, aya_base_tok, MAX_LENGTH)
old_bpb = compute_model_bpb(aya_base, old_ds, count_utf8_bytes(short_texts), torch).bits_per_byte
full_bpb = compute_model_bpb_full(aya_base, short_texts, aya_base_tok, torch, CHUNK_SIZE)["bits_per_byte"]
print(f"old (library)     : {old_bpb:.6f}")
print(f"full (this helper): {full_bpb:.6f}")
assert abs(old_bpb - full_bpb) < 1e-3, "full-coverage helper diverged from the library metric on non-truncated texts"
print("OK - helper matches the library metric when nothing truncates.")

## E. Corrected BASE BPB (full byte coverage, no adapters)

Base tiny-aya BPB: **original** truncating (full 2,500 rows) vs **corrected** full-coverage.
Headline check: does the base BPB rise substantially from 0.8779?

In [ ]:
_set_model_token_config(aya_base, aya_base_tok)  # defensive re-align in case cell is re-run standalone
old_full = compute_model_bpb(
    aya_base, build_text_dataset(eval_texts, aya_base_tok, MAX_LENGTH), EVAL_BYTES, torch
).bits_per_byte
corrected_base = compute_model_bpb_full(aya_base, eval_texts, aya_base_tok, torch, CHUNK_SIZE)["bits_per_byte"]
base_trunc_2500 = old_full  # retained for row-count-matched attribution in section G
del aya_base
free()

base_df = pd.DataFrame({"tiny-aya-base": {
    "old_trunc_full2500": round(old_full, 4),
    "corrected_full": round(corrected_base, 4),
    "delta": round(corrected_base - old_full, 4),
}}).T
print("Base BPB: original (truncating) vs corrected (full coverage), full 2,500 rows")
base_df

## F. Retrain the tiny-aya Akan adapter in-session (both arms)

Reproduce the 2A5 adapters from code (seed 42) — no Kaggle retrieval. Same hyperparameters as the
2A5 run. Adapters go to `*_helper` dirs so they never collide with the committed run.
(`CohereLabs/tiny-aya-base` is already on the `colab-qlora` allowlist.)

First a pre-flight: confirm the LoRA target leaf names match the defaults
(`q/k/v/o_proj`, `gate/up/down_proj`). If tiny-aya uses different names, add
`--lora-target-modules <comma,list>` to both run cells below.

In [ ]:
# Pre-flight: inspect tiny-aya-base projection leaves.
# Uses torch.device("meta") to allocate an empty-weight model (no parameters loaded),
# avoiding a full 3.35B-parameter load just to read module names.
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(AYA_ID)
print("vocab_size         :", getattr(cfg, "vocab_size", None))
print("tie_word_embeddings:", getattr(cfg, "tie_word_embeddings", None))
print("model_type         :", getattr(cfg, "model_type", None))
with torch.device("meta"):
    _m = AutoModelForCausalLM.from_config(cfg)
leaves = sorted({n.split(".")[-1] for n, _ in _m.named_modules() if n.endswith("_proj")})
print("projection leaves  :", leaves)
del _m

In [ ]:
# Arm A - random embedding init
!python scripts/model_integration.py \
    --experiment-id phase2a5_tiny_aya_base_tts_helper \
    --model-id CohereLabs/tiny-aya-base \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a5_tiny_aya_base_tts_helper \
    --results-output results/phase2a5_tiny_aya_base_tts_helper.json \
    --device-mode colab-qlora \
    --embedding-init-mode random \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

In [ ]:
# Arm B - mean-of-subword embedding init
!python scripts/model_integration.py \
    --experiment-id phase2a5_tiny_aya_base_tts_meansub_helper \
    --model-id CohereLabs/tiny-aya-base \
    --tokenizer-path models/tts_tokenizer.json \
    --train-file data/pristine_twi_train.jsonl \
    --eval-file data/pristine_twi_test.jsonl \
    --output-dir models/phase2a5_tiny_aya_base_tts_meansub_helper \
    --results-output results/phase2a5_tiny_aya_base_tts_meansub_helper.json \
    --device-mode colab-qlora \
    --embedding-init-mode mean_subword \
    --max-train-samples 4096 \
    --max-eval-samples 512 \
    --max-length 256 \
    --batch-size 1 \
    --grad-accum 16 \
    --epochs 1 \
    --learning-rate 2e-4 \
    --lora-r 16

## G. Corrected EXPERIMENT BPB + corrected improvement

Reload each saved adapter following the same sequence as `verify_saved_qwen_artifacts`
(resize embeddings, then `PeftModel.from_pretrained`, with the resized `embed_tokens`/`lm_head`
restored via `modules_to_save`). Score the fine-tuned model with full byte coverage using its own
Akan tokenizer, then compute the corrected `base - experiment` against the corrected tiny-aya base
BPB from cell E.

In [ ]:
def reload_adapter(output_dir, model_id):
    """Mirror akan_bpe.model_integration.verify_saved_qwen_artifacts reload sequence."""
    tok = AutoTokenizer.from_pretrained(output_dir)
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token
    qc = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    base = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=qc, dtype="auto")
    base.config.tie_word_embeddings = False
    _set_model_token_config(base, tok)
    base.resize_token_embeddings(len(tok), pad_to_multiple_of=64)
    model = PeftModel.from_pretrained(base, output_dir)
    _set_model_token_config(model, tok)
    model.eval()
    return model, tok


ARMS = {
    "random": "models/phase2a5_tiny_aya_base_tts_helper",
    "mean_subword": "models/phase2a5_tiny_aya_base_tts_meansub_helper",
}

exp_bpb = {}
for arm, out_dir in ARMS.items():
    model, tok = reload_adapter(out_dir, AYA_ID)
    corrected = compute_model_bpb_full(model, eval_texts, tok, torch, CHUNK_SIZE)["bits_per_byte"]
    # Row-count-matched truncating BPB: same metric as the original run but on all 2,500 rows.
    # Holding row count fixed isolates coverage correction from sample-size change.
    trunc_2500 = compute_model_bpb(
        model, build_text_dataset(eval_texts, tok, MAX_LENGTH), EVAL_BYTES, torch
    ).bits_per_byte
    exp_bpb[arm] = {
        "experiment_corrected": round(corrected, 4),
        "experiment_trunc_2500": round(trunc_2500, 4),
        "base_corrected": round(corrected_base, 4),
        "base_trunc_2500": round(base_trunc_2500, 4),
        "improvement_corrected": round(corrected_base - corrected, 4),
        "improvement_trunc_2500": round(base_trunc_2500 - trunc_2500, 4),
    }
    del model
    free()

exp_df = pd.DataFrame(exp_bpb).T
print("Corrected experiment BPB (full coverage) and corrected improvement vs base")
exp_df

## H. Summary — original vs truncation-corrected

Reported (512-row, truncating) 2A5 numbers vs corrected (full 2,500-row, full-coverage), with the
verdict on whether tiny-aya flips positive.

In [ ]:
# Reported 2A5 numbers (report section 12; 512-row capped, truncating BPB).
REPORTED = {
    "base":         0.8779,
    "random":       {"experiment": 1.0766, "improvement": -0.1988},
    "mean_subword": {"experiment": 0.8949, "improvement": -0.0170},
}

summary = []
for arm in ("random", "mean_subword"):
    summary.append({
        "arm": arm,
        # reported: 512 rows, truncating metric
        "base_reported":    REPORTED["base"],
        "exp_reported":     REPORTED[arm]["experiment"],
        "improv_reported":  REPORTED[arm]["improvement"],
        # trunc/2500: truncating metric, 2500 rows — isolates row-count effect from coverage fix
        "base_trunc_2500":  exp_bpb[arm]["base_trunc_2500"],
        "exp_trunc_2500":   exp_bpb[arm]["experiment_trunc_2500"],
        "improv_trunc_2500": exp_bpb[arm]["improvement_trunc_2500"],
        # full/2500: full coverage, 2500 rows — the corrected result
        "base_corrected":   exp_bpb[arm]["base_corrected"],
        "exp_corrected":    exp_bpb[arm]["experiment_corrected"],
        "improv_corrected": exp_bpb[arm]["improvement_corrected"],
    })
summary_df = pd.DataFrame(summary).set_index("arm")
print("2A5 tiny-aya: reported (trunc/512) → trunc/2500 → full-coverage/2500")
print(summary_df.to_string())

flipped_trunc = exp_bpb["mean_subword"]["improvement_trunc_2500"] > 0
flipped = exp_bpb["mean_subword"]["improvement_corrected"] > 0
print()
print(f"mean_subword trunc/2500   : {exp_bpb['mean_subword']['improvement_trunc_2500']:+.4f}"
      f"  ->  {'flips positive (row-count alone sufficient)' if flipped_trunc else 'still negative — row count not the cause'}")
print(f"mean_subword full-coverage: {exp_bpb['mean_subword']['improvement_corrected']:+.4f}"
      f"  ->  {'FLIPS POSITIVE (Akan wins after correction)' if flipped else 'still negative'}")
print()
print("If improv_trunc_2500 stays negative but improv_corrected flips positive, the flip is")
print("attributable to coverage correction alone — not to the change in eval-set size.")
print()
print("Pairs with notebooks/2a4_llama_eval_helper.ipynb: tiny-aya and Llama are the ladder's two")
print("highest-base-fertility rungs and the two with a negative raw sign. If both flip positive")
print("under full-coverage scoring, the negatives were a metric artifact - fix build_text_dataset /")
print("compute_model_bpb in the library and re-run the ladder.")

### Optional cross-checks

Two independent robustness checks for the corrected BPB result:

1. **Chunk size 256 vs 512** (`compute_model_bpb_full`): halving boundary density tests whether
   the chunk cold-start changes the BPB quantitatively.
2. **Sliding window** (`compute_model_bpb_sliding`, window=512, stride=256): eliminates cold-start
   bias entirely by giving every new target ≥255 tokens of left context. If the sliding-window
   improvement agrees in *sign* with the non-overlapping result, boundary bias is not driving
   the flip.

In [ ]:
# --- Cross-check 1: chunk_size 256 vs 512 ---
aya_base2 = _load_base_model_for_bpb(AYA_ID, AutoModelForCausalLM, torch, quantized=True)
_set_model_token_config(aya_base2, aya_base_tok)
for cs in (256, 512):
    r = compute_model_bpb_full(aya_base2, eval_texts, aya_base_tok, torch, chunk_size=cs)
    print(f"corrected base tiny-aya BPB @ chunk_size={cs}: {r['bits_per_byte']:.4f}")
del aya_base2
free()

# --- Cross-check 2: sliding window (window=512, stride=256) ---
# Every new target has >=255 tokens of left context within its window — no cold-start.
# Reloads tiny-aya base + mean_subword adapter once more (~same cost as cross-check 1).
print()
print("Sliding-window confirmation (window=512, stride=256) ...")
aya_base3 = _load_base_model_for_bpb(AYA_ID, AutoModelForCausalLM, torch, quantized=True)
_set_model_token_config(aya_base3, aya_base_tok)
sliding_base = compute_model_bpb_sliding(aya_base3, eval_texts, aya_base_tok, torch)
del aya_base3; free()
print(f"  Base tiny-aya (sliding w=512 s=256)         : {sliding_base['bits_per_byte']:.4f}")
print(f"  Base tiny-aya (non-overlapping chunk=256)   : {corrected_base:.4f}")

model_sw, tok_sw = reload_adapter(ARMS["mean_subword"], AYA_ID)
sliding_exp = compute_model_bpb_sliding(model_sw, eval_texts, tok_sw, torch)
del model_sw; free()
print(f"  Exp mean_subword (sliding)                  : {sliding_exp['bits_per_byte']:.4f}")
print(f"  Exp mean_subword (non-overlapping)          : {exp_bpb['mean_subword']['experiment_corrected']:.4f}")
sliding_improv = sliding_base["bits_per_byte"] - sliding_exp["bits_per_byte"]
print(f"  Improvement (sliding)                       : {sliding_improv:+.4f}")
print(f"  Improvement (non-overlapping)               : {exp_bpb['mean_subword']['improvement_corrected']:+.4f}")
qualitative_agree = (sliding_improv > 0) == (exp_bpb["mean_subword"]["improvement_corrected"] > 0)
print(f"\n  Qualitative agreement: "
      f"{'YES — boundary cold-start is not driving the flip' if qualitative_agree else 'NO — results diverge; chunk-boundary bias may be significant'}")